In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import GRU, Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam

In [3]:
df = pd.read_csv("AirPassengers.csv")

In [7]:
df.sample(10)

,Month,#Passengers
76,1955-05,270
9,1949-10,119
16,1950-05,125
68,1954-09,259
67,1954-08,293
15,1950-04,135
129,1959-10,407
37,1952-02,180
90,1956-07,413
45,1952-10,191


In [8]:
# 2. Select passengers column
data = df["#Passengers"].values.reshape(-1, 1)

In [9]:
# 3. Train-test split
train_size = int(len(data) * 0.8)
train_data = data[:train_size]
test_data = data[train_size:]

In [10]:
# 4. Scaling
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train_data)
test_scaled = scaler.transform(test_data)

In [11]:
# 5. Create sequences
def create_sequences(data, sequence_length):
    X = []
    y = []

    for i in range(len(data) - sequence_length):
        X.append(data[i:i + sequence_length])
        y.append(data[i + sequence_length])

    return np.array(X), np.array(y)


sequence_length = 12

X_train, y_train = create_sequences(
    train_scaled,
    sequence_length
)

X_test, y_test = create_sequences(
    test_scaled,
    sequence_length
)


In [12]:
# 6. Check shapes
print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (103, 12, 1)
y_train: (103, 1)
X_test: (17, 12, 1)
y_test: (17, 1)


In [13]:
# 7. Build GRU model
model = Sequential([
    GRU(
        64,
        input_shape=(sequence_length, 1)
    ),

    Dropout(0.2),

    Dense(32, activation="relu"),

    Dense(1)
])

C:\ProgramData\anaconda3\Lib\site-packages\keras\src\layers\rnn\rnn.py:199: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(**kwargs)


In [14]:
# 8. Compile
model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

In [15]:
# 9. Train
history = model.fit(
    X_train,
    y_train,
    epochs=100,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

Epoch 1/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 3s 86ms/step - loss: 0.0849 - mae: 0.2399 - val_loss: 0.1351 - val_mae: 0.3396
Epoch 2/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 18ms/step - loss: 0.0203 - mae: 0.1025 - val_loss: 0.0253 - val_mae: 0.1205
Epoch 3/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0140 - mae: 0.1024 - val_loss: 0.0199 - val_mae: 0.1134
Epoch 4/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms/step - loss: 0.0126 - mae: 0.0930 - val_loss: 0.0257 - val_mae: 0.1217
Epoch 5/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0093 - mae: 0.0691 - val_loss: 0.0405 - val_mae: 0.1642
Epoch 6/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 17ms/step - loss: 0.0092 - mae: 0.0694 - val_loss: 0.0315 - val_mae: 0.1407
Epoch 7/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 0.0086 - mae: 0.0736 - val_loss: 0.0220 - val_mae: 0.1135
Epoch 8/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0071 - mae: 0.0651 - val_loss: 0.0223 - val_mae: 0.1143
Epoch 9/100
6/6 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - loss: 0.0085 - mae: 

In [16]:
# 10. Predict
predictions = model.predict(X_test)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 167ms/step


In [17]:
# 11. Convert back to original scale
predictions = scaler.inverse_transform(predictions)
actual = scaler.inverse_transform(y_test)

In [18]:
# 12. Display results
for a, p in zip(actual[:10], predictions[:10]):
    print(
        f"Actual: {a[0]:.0f}, "
        f"Predicted: {p[0]:.0f}"
    )

Actual: 559, Predicted: 522
Actual: 463, Predicted: 516
Actual: 407, Predicted: 426
Actual: 362, Predicted: 401
Actual: 405, Predicted: 371
Actual: 417, Predicted: 418
Actual: 391, Predicted: 420
Actual: 419, Predicted: 395
Actual: 461, Predicted: 425
Actual: 472, Predicted: 458
